# GreenSplit – PyTorch cross-check and re-run

Run this notebook **from the package root** (the folder that contains `greensplit/`, `original/`, `results/`).
It does three things:

1. Measures per-block output tensor sizes and MAC counts of MobileNetV2, ResNet-50, ViT-B/16 and ViT-B/32 with PyTorch/torchvision (random weights are fine: sizes and MACs do not depend on weights) and compares them with the analytic profiles in `greensplit/sim.py` that every experiment in the paper uses.
2. Re-runs `greensplit/experiments.py`, `extra_analysis.py` and `privacy_ttest.py` and checks that the numbers match the stored `results/results.json`.
3. Writes `results/torch_check.json`. **Send that file (or the output of the last cell) back** so the manuscript statement that the analytic profiles were checked against PyTorch can be confirmed for all four architectures.

Needs: `torch`, `torchvision` (CPU is enough), `numpy`, `scipy`, `matplotlib`.

In [ ]:
import sys, subprocess, importlib
for pkg in ['torch', 'torchvision']:
    try: importlib.import_module(pkg)
    except ImportError: subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg])
import os, json, numpy as np, torch, torchvision
assert os.path.isdir('greensplit'), 'run from the package root'
sys.path.insert(0, os.getcwd())
from greensplit import sim
print(torch.__version__, torchvision.__version__)

## 1. Block definitions (the split granularity used in the paper)

MobileNetV2: `features[0..18]` (19 blocks). ResNet-50: the 16 bottleneck blocks (the stem always runs on the device and is added to block 0's MACs in `sim.py`). ViT-B/16 and ViT-B/32: the 12 encoder layers (patch embedding is added to layer 0). Block 0 is therefore excluded from the per-block MAC comparison and only compared through the totals.

In [ ]:
from torch.utils.flop_counter import FlopCounterMode
from torchvision import models
torch.manual_seed(0)
torch.backends.mha.set_fastpath_enabled(False)   # the fused CPU fast path is invisible to the FLOP counter

def blocks_of(name):
    if name == 'MobileNetV2':
        m = models.mobilenet_v2(weights=None).eval(); return m, list(m.features)
    if name == 'ResNet-50':
        m = models.resnet50(weights=None).eval()
        return m, [b for l in (m.layer1, m.layer2, m.layer3, m.layer4) for b in l]
    if name == 'ViT-Base':
        m = models.vit_b_16(weights=None).eval(); return m, list(m.encoder.layers)
    if name == 'CLIP ViT-B/32':      # image tower of CLIP ViT-B/32 has the torchvision vit_b_32 geometry (50 tokens, width 768)
        m = models.vit_b_32(weights=None).eval(); return m, list(m.encoder.layers)

def profile(name):
    m, blocks = blocks_of(name)
    ins, outs = {}, {}
    def mk(i):
        def hook(mod, inp, out): ins[i] = inp[0].detach(); outs[i] = out.detach()
        return hook
    hs = [b.register_forward_hook(mk(i)) for i, b in enumerate(blocks)]
    x = torch.randn(1, 3, 224, 224)
    with torch.no_grad(): m(x)
    for h in hs: h.remove()
    elems = np.array([outs[i].numel() for i in range(len(blocks))])
    macs = []
    for i, b in enumerate(blocks):
        with torch.no_grad(), FlopCounterMode(display=False) as fc: b(ins[i])
        macs.append(fc.get_total_flops() / 2 / 1e6)          # FLOPs -> MACs
    macs = np.array(macs)
    with torch.no_grad(), FlopCounterMode(display=False) as fc_all: m(x)
    total = fc_all.get_total_flops() / 2 / 1e6                # whole network incl. stem/patch-embed/head
    return elems, macs, total

out = {}
for name in sim.ARCH_NAMES:
    e_t, m_t, tot_t = profile(name)
    e_s, m_s = sim.profile_arrays(name)
    same_len = len(e_t) == len(e_s)
    size_ok = bool(same_len and np.all(e_t == e_s))
    rel = np.abs(m_t[1:] - m_s[1:]) / np.maximum(m_s[1:], 1e-9) if same_len else np.array([np.nan])
    out[name] = dict(n_blocks_torch=int(len(e_t)), n_blocks_sim=int(len(e_s)), sizes_identical=size_ok,
                     total_mmac_torch_full_network=float(tot_t), total_mmac_sim_blocks=float(m_s.sum()),
                     max_rel_err_blocks_1_to_end=float(rel.max()),
                     elems_first3_torch=e_t[:3].tolist(), elems_first3_sim=[int(v) for v in e_s[:3]])
    print(name, json.dumps(out[name]))

**What to expect.** `sizes_identical` must be `True` for all four architectures (it is a pure shape computation). Per-block MACs should agree to within a few percent; the torch counter and the analytic count both ignore element-wise layers, but attention is counted slightly differently. Report any larger difference: it does not change the transmitted tensor sizes, only the compute-time split between device and cloud.

In [ ]:
assert all(v['sizes_identical'] for v in out.values()), 'tensor-size mismatch: report this'
print('tensor sizes identical for', list(out))
for k, v in out.items():
    r = abs(v['total_mmac_torch_full_network'] - v['total_mmac_sim_blocks']) / v['total_mmac_sim_blocks']
    print(f'{k}: total MACs torch {v["total_mmac_torch_full_network"]:.0f} M vs analytic {v["total_mmac_sim_blocks"]:.0f} M ({100*r:.1f} % difference)')

## 2. Re-run the experiments and compare with the stored results

In [ ]:
import shutil
os.makedirs('results', exist_ok=True)
shutil.copy('results/results.json', 'results/results_stored.json')
RUN_PRIVACY_IMAGENET = False  # set True to rerun the ImageNet evaluation (Sec. 6.5(D), ~20 min, 8 GB RAM). Needs the 1000-image sample:
#   git clone --depth 1 https://github.com/EliSchwartz/imagenet-sample-images data/imagenet_sample      (checkpoint is already in data/)
RUN_PRIVACY_REAL = False   # set True to also retrain the MNIST network and rerun the real-activation attacks (~5 min, needs mlxtend + scikit-learn)
for s in ['experiments.py', 'extra_analysis.py', 'privacy_ttest.py', 'thermal_sensitivity.py', 'awc_study.py'] + (['privacy_real.py'] if RUN_PRIVACY_REAL else []) + (['privacy_imagenet.py'] if RUN_PRIVACY_IMAGENET else []):
    subprocess.check_call([sys.executable, 'greensplit/' + s])
new = json.load(open('results/results.json')); old = json.load(open('results/results_stored.json'))
def diff(a, b, path=''):
    bad = []
    if isinstance(a, dict):
        for k in a: bad += diff(a[k], b.get(k) if isinstance(b, dict) else None, path + '/' + str(k))
    elif isinstance(a, (int, float)) and isinstance(b, (int, float)):
        if abs(a - b) > 1e-9 * max(1, abs(b)): bad.append((path, a, b))
    elif a != b: bad.append((path, a, b))
    return bad
bad = diff(new, old)
print('numbers that differ from the stored run:', len(bad)); print(bad[:10])
print({k: round(v['lat_mean'], 1) for k, v in new['table9'].items()})

## 3. Summary to send back

In [ ]:
out['rerun_matches_stored_results'] = (len(bad) == 0)
out['torch'] = torch.__version__; out['torchvision'] = torchvision.__version__
json.dump(out, open('results/torch_check.json', 'w'), indent=1)
print(json.dumps(out, indent=1))